# FinLora Fraud Detection Project
## 01 — Data Understanding

### Objective

The objective of this notebook is to understand the structure and content of the FinLora transaction dataset before proceeding to data profiling, cleaning, feature engineering, and machine learning model development.

The analysis will examine:

- Dataset structure and dimensions
- Available variables and their data types
- Numerical, categorical, temporal, identifier, and target variables
- Potential keys and relationships
- The fraud target variable
- Potentially relevant fraud-detection features
- Variables requiring further investigation before modelling

The findings from this notebook will provide the foundation for subsequent data profiling and data preparation activities.

In [5]:
from pathlib import Path

import pandas as pd
import numpy as np

In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "Dataset"
    / "raw"
    / "FinLora_Customer_Transaction_Dataset.csv"
)

df = pd.read_csv(DATA_PATH)

df.head()

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [6]:
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

Number of rows: 11,400
Number of columns: 26


In [7]:
df.columns.tolist()

['transaction_id',
 'customer_id',
 'timestamp',
 'home_country',
 'source_currency',
 'dest_currency',
 'channel',
 'amount_src',
 'amount_usd',
 'fee',
 'exchange_rate_src_to_dest',
 'device_id',
 'new_device',
 'ip_address',
 'ip_country',
 'location_mismatch',
 'ip_risk_score',
 'kyc_tier',
 'account_age_days',
 'device_trust_score',
 'chargeback_history_count',
 'risk_score_internal',
 'txn_velocity_1h',
 'txn_velocity_24h',
 'corridor_risk',
 'is_fraud']

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  object 
 1   customer_id                11400 non-null  object 
 2   timestamp                  11371 non-null  object 
 3   home_country               11400 non-null  object 
 4   source_currency            11400 non-null  object 
 5   dest_currency              11400 non-null  object 
 6   channel                    11400 non-null  object 
 7   amount_src                 11400 non-null  object 
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  object 
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  obj

## 1. Dataset Structure

The FinLora dataset contains 11,400 transaction records and 26 variables. Each row represents a transaction-level observation containing information relating to the transaction, customer, device, geographic location, behavioural characteristics, risk indicators, and fraud classification.

The dataset is provided as a single denormalized table rather than as separate transaction, customer, and fraud-investigation tables. Therefore, customer and transaction attributes are contained within the same dataset and no relational joins are required at this stage.

The dataset contains the following variable types:

- 12 object/string variables
- 7 floating-point numerical variables
- 5 integer variables
- 2 Boolean variables

The initial inspection also identified that some variables require data-type validation. In particular, `timestamp` is currently stored as an object rather than a datetime type, while `amount_src` is stored as an object despite representing a transaction amount. These issues will be investigated during the data cleaning stage.

## 2. Variable Inventory

The following section examines the variables available in the FinLora dataset and classifies them according to their likely analytical role.

In [9]:
df.columns.tolist()

['transaction_id',
 'customer_id',
 'timestamp',
 'home_country',
 'source_currency',
 'dest_currency',
 'channel',
 'amount_src',
 'amount_usd',
 'fee',
 'exchange_rate_src_to_dest',
 'device_id',
 'new_device',
 'ip_address',
 'ip_country',
 'location_mismatch',
 'ip_risk_score',
 'kyc_tier',
 'account_age_days',
 'device_trust_score',
 'chargeback_history_count',
 'risk_score_internal',
 'txn_velocity_1h',
 'txn_velocity_24h',
 'corridor_risk',
 'is_fraud']

In [10]:
variable_classification = pd.DataFrame({
    "Variable": [
        "transaction_id",
        "customer_id",
        "timestamp",
        "home_country",
        "source_currency",
        "dest_currency",
        "channel",
        "amount_src",
        "amount_usd",
        "fee",
        "exchange_rate_src_to_dest",
        "device_id",
        "new_device",
        "ip_address",
        "ip_country",
        "location_mismatch",
        "ip_risk_score",
        "kyc_tier",
        "account_age_days",
        "device_trust_score",
        "chargeback_history_count",
        "risk_score_internal",
        "txn_velocity_1h",
        "txn_velocity_24h",
        "corridor_risk",
        "is_fraud"
    ],
    
    "Variable_Type": [
        "Identifier",
        "Identifier",
        "Datetime",
        "Categorical",
        "Categorical",
        "Categorical",
        "Categorical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Identifier",
        "Binary",
        "Identifier",
        "Categorical",
        "Binary",
        "Numerical",
        "Categorical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Target"
    ]
})

variable_classification

,Variable,Variable_Type
0,transaction_id,Identifier
1,customer_id,Identifier
2,timestamp,Datetime
3,home_country,Categorical
4,source_currency,Categorical
5,dest_currency,Categorical
6,channel,Categorical
7,amount_src,Numerical
8,amount_usd,Numerical
9,fee,Numerical


## 3. Target Variable

The target variable for the fraud detection problem is `is_fraud`.

This variable represents the historical fraud classification associated with each transaction. It is encoded as a binary integer variable:

- `0` = legitimate transaction
- `1` = fraudulent transaction

The `is_fraud` variable will serve as the dependent/target variable for the supervised machine learning classification models developed later in the project.

In [12]:
df["is_fraud"].value_counts()

is_fraud
0    10403
1      997
Name: count, dtype: int64

In [13]:
df["is_fraud"].value_counts(normalize=True) * 100

is_fraud
0    91.254386
1     8.745614
Name: proportion, dtype: float64

In [14]:
df["is_fraud"].unique()

array([0, 1])

In [15]:
df["is_fraud"].isna().sum()

np.int64(0)

## 4. Identifiers and Potential Keys

Identifier variables are examined to determine their uniqueness and potential role in the data structure. The main identifiers available are `transaction_id`, `customer_id`, `device_id`, and `ip_address`.

`transaction_id` is expected to identify individual transaction records and is therefore examined for uniqueness. `customer_id`, `device_id`, and `ip_address` may occur across multiple transactions and can provide useful information for behavioural feature engineering.

In [16]:
df["transaction_id"].nunique()

11200

In [17]:
df["transaction_id"].duplicated().sum()

np.int64(200)

In [18]:
df["transaction_id"].value_counts().head(10)

transaction_id
805e3831-dd30-44cd-9db8-122425ec3a36    2
5263bbdf-1a0a-48a3-8309-3f6b5b57d4a6    2
ba28f0a4-1ee4-413a-b371-11449236ed6f    2
cc815d51-f920-4314-a84a-28730c76bd08    2
35a684af-4d53-4ca4-a41a-5409c9e209c7    2
ade63e2f-ea83-4e26-88ee-3f89dce7b465    2
ceea2014-a8f4-40d1-a745-3e647ca93da7    2
d1849b01-2aef-47af-9363-6eb542a11203    2
b85d6925-a73c-4008-a9ac-f0b179bc8d43    2
e952671a-33bc-4c98-9242-f47caad7741c    2
Name: count, dtype: int64

In [36]:
duplicate_transaction_ids = (
    df[df["transaction_id"].duplicated(keep=False)]
    .sort_values("transaction_id")
)

duplicate_transaction_ids[
    ["transaction_id", "customer_id", "timestamp", "is_fraud"]
].head(20)

,transaction_id,customer_id,timestamp,is_fraud
10153,03368cab-696b-4984-bffb-3429267e57b2,7bd5200c-5d19-44f0-9afe-8b339a05366b,2024-06-29 04:15:54.468549+00:00,0
5866,03368cab-696b-4984-bffb-3429267e57b2,7bd5200c-5d19-44f0-9afe-8b339a05366b,2024-06-29 04:15:54.468549+00:00,0
6264,0368fc3c-2323-470e-aafe-eb0d920553eb,d71c91b4-fee8-4104-9856-a5c6109a62e3,2024-08-16 14:03:44.468549+00:00,0
10079,0368fc3c-2323-470e-aafe-eb0d920553eb,d71c91b4-fee8-4104-9856-a5c6109a62e3,2024-08-16 14:03:44.468549+00:00,0
5185,04532993-f980-4c87-9129-373a00ed57db,d71c91b4-fee8-4104-9856-a5c6109a62e3,2024-04-18 10:26:16.468549+00:00,0
10036,04532993-f980-4c87-9129-373a00ed57db,d71c91b4-fee8-4104-9856-a5c6109a62e3,2024-04-18 10:26:16.468549+00:00,0
10085,061c3f56-bff6-47d9-b195-1ecdc1faee8a,af8ca4c4-8703-4c55-b66c-2b76cd70040d,2024-10-06 05:17:22.468549+00:00,0
6700,061c3f56-bff6-47d9-b195-1ecdc1faee8a,af8ca4c4-8703-4c55-b66c-2b76cd70040d,2024-10-06 05:17:22.468549+00:00,0
10136,068abac2-c06a-42af-9a38-379886b26e4a,d71c91b4-fee8-4104-9856-a5c6109a62e3,2023-07-14 03:58:42.468549+00:00,0
2572,068abac2-c06a-42af-9a38-379886b26e4a,d71c91b4-fee8-4104-9856-a5c6109a62e3,2023-07-14 03:58:42.468549+00:00,0


In [32]:
duplicate_transaction_ids["transaction_id"].value_counts().value_counts()

count
2    200
Name: count, dtype: int64

In [34]:
duplicate_transaction_ids.shape

(400, 26)

In [38]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 200


In [39]:
duplicate_comparison = (
    duplicate_transaction_ids
    .groupby("transaction_id")
    .nunique()
)

duplicate_comparison.eq(1).all(axis=1).value_counts()

True     195
False      5
Name: count, dtype: int64

In [40]:
# Identify transaction IDs that occur twice but contain different information
non_identical_duplicate_ids = (
    duplicate_comparison[
        ~duplicate_comparison.eq(1).all(axis=1)
    ].index
)

non_identical_duplicate_records = (
    df[df["transaction_id"].isin(non_identical_duplicate_ids)]
    .sort_values("transaction_id")
)

non_identical_duplicate_records

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
4741,356a1a21-e76c-4ad6-b712-405d3ad215e6,23d9943d-57b1-42eb-9ef2-05eed7fd1957,2024-03-04 21:11:40.468549+00:00,UK,GBP,INR,mobile,83.96,NaN,NaN,...,0.474,NaN,718,NaN,0,0.169,0,0,0.0,0
10142,356a1a21-e76c-4ad6-b712-405d3ad215e6,23d9943d-57b1-42eb-9ef2-05eed7fd1957,2024-03-04 21:11:40.468549+00:00,UK,GBP,INR,mobile,83.96,NaN,NaN,...,0.474,NaN,718,NaN,0,0.169,0,0,0.0,0
9037,35da0a3e-5776-4f7e-b102-5ea3d5034713,402cccc9-28de-45b3-9af7-cc5302aa1f93,2025-06-20 07:38:53.468549+00:00,US,USD,EUR,mobile,9993.94,NaN,NaN,...,0.434,NaN,263,NaN,0,0.223,0,0,0.0,0
10054,35da0a3e-5776-4f7e-b102-5ea3d5034713,402cccc9-28de-45b3-9af7-cc5302aa1f93,2025-06-20 07:38:53.468549+00:00,US,USD,EUR,mobile,9993.94,NaN,NaN,...,0.434,NaN,263,NaN,0,0.223,0,0,0.0,0
7714,adc74103-d3b7-491e-a3c9-02aae2050d46,7bd5200c-5d19-44f0-9afe-8b339a05366b,2025-01-27 17:14:38.468549+00:00,US,USD,USD,mobile,414.29,NaN,NaN,...,0.446,NaN,147,NaN,0,0.391,0,0,0.0,1
10082,adc74103-d3b7-491e-a3c9-02aae2050d46,7bd5200c-5d19-44f0-9afe-8b339a05366b,2025-01-27 17:14:38.468549+00:00,US,USD,USD,mobile,414.29,NaN,NaN,...,0.446,NaN,147,NaN,0,0.391,0,0,0.0,1
5852,d5ce08b7-d78f-4e1b-97c0-362cc64e12ee,af8ca4c4-8703-4c55-b66c-2b76cd70040d,2024-06-28 06:10:20.468549+00:00,US,USD,PHP,mobile,125.75,NaN,NaN,...,0.451,NaN,1018,NaN,0,0.087,0,0,0.1,0
10093,d5ce08b7-d78f-4e1b-97c0-362cc64e12ee,af8ca4c4-8703-4c55-b66c-2b76cd70040d,2024-06-28 06:10:20.468549+00:00,US,USD,PHP,mobile,125.75,NaN,NaN,...,0.451,NaN,1018,NaN,0,0.087,0,0,0.1,0
2595,ddb073fa-05d9-45b0-90bf-7336b7ef67ff,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2023-07-16 23:39:30.468549+00:00,US,USD,EUR,mobile,392.41,NaN,NaN,...,0.332,NaN,367,NaN,0,0.176,0,0,0.0,0
10162,ddb073fa-05d9-45b0-90bf-7336b7ef67ff,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2023-07-16 23:39:30.468549+00:00,US,USD,EUR,mobile,392.41,NaN,NaN,...,0.332,NaN,367,NaN,0,0.176,0,0,0.0,0


In [41]:
non_identical_duplicate_records.T

,4741,10142,9037,10054,7714,10082,5852,10093,2595,10162
transaction_id,356a1a21-e76c-4ad6-b712-405d3ad215e6,356a1a21-e76c-4ad6-b712-405d3ad215e6,35da0a3e-5776-4f7e-b102-5ea3d5034713,35da0a3e-5776-4f7e-b102-5ea3d5034713,adc74103-d3b7-491e-a3c9-02aae2050d46,adc74103-d3b7-491e-a3c9-02aae2050d46,d5ce08b7-d78f-4e1b-97c0-362cc64e12ee,d5ce08b7-d78f-4e1b-97c0-362cc64e12ee,ddb073fa-05d9-45b0-90bf-7336b7ef67ff,ddb073fa-05d9-45b0-90bf-7336b7ef67ff
customer_id,23d9943d-57b1-42eb-9ef2-05eed7fd1957,23d9943d-57b1-42eb-9ef2-05eed7fd1957,402cccc9-28de-45b3-9af7-cc5302aa1f93,402cccc9-28de-45b3-9af7-cc5302aa1f93,7bd5200c-5d19-44f0-9afe-8b339a05366b,7bd5200c-5d19-44f0-9afe-8b339a05366b,af8ca4c4-8703-4c55-b66c-2b76cd70040d,af8ca4c4-8703-4c55-b66c-2b76cd70040d,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,6d0d9b27-fa26-45f8-93b1-2df29d182d9c
timestamp,2024-03-04 21:11:40.468549+00:00,2024-03-04 21:11:40.468549+00:00,2025-06-20 07:38:53.468549+00:00,2025-06-20 07:38:53.468549+00:00,2025-01-27 17:14:38.468549+00:00,2025-01-27 17:14:38.468549+00:00,2024-06-28 06:10:20.468549+00:00,2024-06-28 06:10:20.468549+00:00,2023-07-16 23:39:30.468549+00:00,2023-07-16 23:39:30.468549+00:00
home_country,UK,UK,US,US,US,US,US,US,US,US
source_currency,GBP,GBP,USD,USD,USD,USD,USD,USD,USD,USD
dest_currency,INR,INR,EUR,EUR,USD,USD,PHP,PHP,EUR,EUR
channel,mobile,mobile,mobile,mobile,mobile,mobile,mobile,mobile,mobile,mobile
amount_src,83.96,83.96,9993.94,9993.94,414.29,414.29,125.75,125.75,392.41,392.41
amount_usd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fee,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
non_identical_duplicate_records[
    [
        "transaction_id",
        "customer_id",
        "timestamp",
        "amount_src",
        "amount_usd",
        "fee",
        "device_id",
        "ip_address",
        "ip_country",
        "is_fraud"
    ]
]

,transaction_id,customer_id,timestamp,amount_src,amount_usd,fee,device_id,ip_address,ip_country,is_fraud
4741,356a1a21-e76c-4ad6-b712-405d3ad215e6,23d9943d-57b1-42eb-9ef2-05eed7fd1957,2024-03-04 21:11:40.468549+00:00,83.96,NaN,NaN,1bbcba69-0f10-499b-9d26-3d9b59cc3dfc,NaN,NaN,0
10142,356a1a21-e76c-4ad6-b712-405d3ad215e6,23d9943d-57b1-42eb-9ef2-05eed7fd1957,2024-03-04 21:11:40.468549+00:00,83.96,NaN,NaN,1bbcba69-0f10-499b-9d26-3d9b59cc3dfc,NaN,NaN,0
9037,35da0a3e-5776-4f7e-b102-5ea3d5034713,402cccc9-28de-45b3-9af7-cc5302aa1f93,2025-06-20 07:38:53.468549+00:00,9993.94,NaN,NaN,b3290ccd-926f-4e98-b219-eab283ad718a,NaN,NaN,0
10054,35da0a3e-5776-4f7e-b102-5ea3d5034713,402cccc9-28de-45b3-9af7-cc5302aa1f93,2025-06-20 07:38:53.468549+00:00,9993.94,NaN,NaN,b3290ccd-926f-4e98-b219-eab283ad718a,NaN,NaN,0
7714,adc74103-d3b7-491e-a3c9-02aae2050d46,7bd5200c-5d19-44f0-9afe-8b339a05366b,2025-01-27 17:14:38.468549+00:00,414.29,NaN,NaN,eb8cac8c-da67-4a9a-84c9-e67242be9cbd,NaN,NaN,1
10082,adc74103-d3b7-491e-a3c9-02aae2050d46,7bd5200c-5d19-44f0-9afe-8b339a05366b,2025-01-27 17:14:38.468549+00:00,414.29,NaN,NaN,eb8cac8c-da67-4a9a-84c9-e67242be9cbd,NaN,NaN,1
5852,d5ce08b7-d78f-4e1b-97c0-362cc64e12ee,af8ca4c4-8703-4c55-b66c-2b76cd70040d,2024-06-28 06:10:20.468549+00:00,125.75,NaN,NaN,991bd49b-7d62-4ac6-81bf-b2e16aa7c0f2,NaN,NaN,0
10093,d5ce08b7-d78f-4e1b-97c0-362cc64e12ee,af8ca4c4-8703-4c55-b66c-2b76cd70040d,2024-06-28 06:10:20.468549+00:00,125.75,NaN,NaN,991bd49b-7d62-4ac6-81bf-b2e16aa7c0f2,NaN,NaN,0
2595,ddb073fa-05d9-45b0-90bf-7336b7ef67ff,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2023-07-16 23:39:30.468549+00:00,392.41,NaN,NaN,645cfb3b-40db-489f-96d2-e8f372001668,NaN,NaN,0
10162,ddb073fa-05d9-45b0-90bf-7336b7ef67ff,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2023-07-16 23:39:30.468549+00:00,392.41,NaN,NaN,645cfb3b-40db-489f-96d2-e8f372001668,NaN,NaN,0


In [45]:
# Identify exactly which columns differ within each of the 5 non-identical duplicate pairs

for transaction_id in non_identical_duplicate_ids:
    records = df[df["transaction_id"] == transaction_id]
    
    print(f"\nTransaction ID: {transaction_id}")
    
    # Compare the two records column by column
    differing_columns = [
        col for col in df.columns
        if records[col].nunique(dropna=False) > 1
    ]
    
    print("Differing columns:")
    print(differing_columns)
    
    print("\nValues:")
    print(records[differing_columns].to_string(index=False))


Transaction ID: 356a1a21-e76c-4ad6-b712-405d3ad215e6
Differing columns:
[]

Values:
Empty DataFrame
Columns: []
Index: [4741, 10142]

Transaction ID: 35da0a3e-5776-4f7e-b102-5ea3d5034713
Differing columns:
[]

Values:
Empty DataFrame
Columns: []
Index: [9037, 10054]

Transaction ID: adc74103-d3b7-491e-a3c9-02aae2050d46
Differing columns:
[]

Values:
Empty DataFrame
Columns: []
Index: [7714, 10082]

Transaction ID: d5ce08b7-d78f-4e1b-97c0-362cc64e12ee
Differing columns:
[]

Values:
Empty DataFrame
Columns: []
Index: [5852, 10093]

Transaction ID: ddb073fa-05d9-45b0-90bf-7336b7ef67ff
Differing columns:
[]

Values:
Empty DataFrame
Columns: []
Index: [2595, 10162]


### Duplicate Transaction ID Observation


The dataset contains 200 transaction IDs that occur twice, resulting in 400 records being associated with repeated transaction IDs.

Further investigation initially suggested that five transaction ID pairs contained differences. However, additional validation showed that these apparent differences were caused by the treatment of missing values during the uniqueness check. When missing values are appropriately accounted for, all 200 duplicated transaction ID pairs contain identical records across the available variables.

Therefore, the dataset contains 200 redundant duplicate rows corresponding to 200 duplicated transaction IDs. These duplicate records will be addressed during the data cleaning stage. No records are removed during the Data Understanding stage.

In [20]:
print("Unique customers:", df["customer_id"].nunique())
print("Total transactions:", len(df))

Unique customers: 1315
Total transactions: 11400


In [ ]:
# This tells us how transaction activity is distributed across customers.

transactions_per_customer = df["customer_id"].value_counts()

transactions_per_customer.describe()

count    1315.000000
mean        8.669202
std        84.766961
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max      1510.000000
Name: count, dtype: float64

In [22]:
transactions_per_customer.head(10)

customer_id
402cccc9-28de-45b3-9af7-cc5302aa1f93    1510
d71c91b4-fee8-4104-9856-a5c6109a62e3    1355
7041b9c1-3719-4ca8-9a6b-811b47cea6c0    1345
6d0d9b27-fa26-45f8-93b1-2df29d182d9c    1066
af8ca4c4-8703-4c55-b66c-2b76cd70040d     915
7bd5200c-5d19-44f0-9afe-8b339a05366b     811
70a93d26-8e3a-4179-900c-a4a7a74d08e5     677
f7531a78-8bbe-4a85-b1e8-a0650edddc72     394
67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad     360
23d9943d-57b1-42eb-9ef2-05eed7fd1957     347
Name: count, dtype: int64

## Device identifier analysis

In [23]:
print("Unique devices:", df["device_id"].nunique())

Unique devices: 2113


In [24]:
transactions_per_device = df["device_id"].value_counts()

transactions_per_device.describe()

count    2113.000000
mean        5.395173
std         9.197616
min         1.000000
25%         1.000000
50%         1.000000
75%         5.000000
max        87.000000
Name: count, dtype: float64

## IP_address analysis

In [25]:
transactions_per_device = df["device_id"].value_counts()

transactions_per_device.describe()

count    2113.000000
mean        5.395173
std         9.197616
min         1.000000
25%         1.000000
50%         1.000000
75%         5.000000
max        87.000000
Name: count, dtype: float64

In [26]:
df["ip_address"].value_counts(dropna=False).head(10)

ip_address
NaN               305
170.231.29.219      2
254.66.148.80       2
221.188.124.29      2
111.195.28.32       2
130.166.12.67       2
161.196.159.39      2
46.164.196.54       2
253.225.229.30      2
141.199.241.52      2
Name: count, dtype: int64

In [28]:
df["ip_address"].nunique()

10900

## 4.1 Identifier Analysis Findings

The identifier analysis shows that `transaction_id` is not unique within the supplied dataset. There are 11,200 unique transaction IDs across 11,400 records, resulting in 200 duplicated transaction IDs. This issue will be investigated further during the data cleaning stage.

The dataset contains 1,315 unique customers, indicating that individual customers are associated with multiple transaction records. Similarly, 2,113 unique devices are present across the 11,400 transactions, suggesting that devices may be reused across transactions.

There are 10,900 unique IP addresses. Since the dataset contains 11,095 non-null IP address records, some IP addresses occur more than once, while 305 records have missing IP address information.

The identifiers are therefore useful for understanding transaction and behavioural relationships and for subsequent feature engineering. However, raw identifier values should not automatically be treated as predictive features because they may cause memorisation or poor generalisation.

## 5. Variable Relevance Assessment

Each variable is assessed according to its business meaning, analytical role, and potential relevance to fraud detection. This assessment is preliminary and will be refined during exploratory analysis, data cleaning, and feature engineering.

In [29]:
variable_relevance = pd.DataFrame({
    "Variable": df.columns,
    
    "Role": [
        "Identifier",
        "Identifier",
        "Datetime",
        "Categorical",
        "Categorical",
        "Categorical",
        "Categorical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Identifier",
        "Binary",
        "Identifier",
        "Categorical",
        "Binary",
        "Numerical",
        "Categorical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Numerical",
        "Target"
    ],
    
    "Modelling_Relevance": [
        "Exclude as direct predictor; retain for transaction tracking",
        "Potentially useful for customer-level feature engineering",
        "Useful for temporal feature engineering",
        "Potential predictor",
        "Potential predictor",
        "Potential predictor",
        "Potential predictor",
        "Potential predictor after type validation",
        "Potential predictor",
        "Potential predictor",
        "Potential predictor",
        "Potentially useful for device-level feature engineering",
        "Potential predictor",
        "Potentially useful for geographic/network feature engineering",
        "Potential predictor",
        "Potential predictor",
        "Potential predictor",
        "Potential predictor",
        "Potential predictor",
        "Potential predictor",
        "Potential predictor",
        "Requires leakage investigation",
        "Potential predictor",
        "Potential predictor",
        "Potential predictor",
        "Target variable"
    ]
})

variable_relevance

,Variable,Role,Modelling_Relevance
0,transaction_id,Identifier,Exclude as direct predictor; retain for transa...
1,customer_id,Identifier,Potentially useful for customer-level feature ...
2,timestamp,Datetime,Useful for temporal feature engineering
3,home_country,Categorical,Potential predictor
4,source_currency,Categorical,Potential predictor
5,dest_currency,Categorical,Potential predictor
6,channel,Categorical,Potential predictor
7,amount_src,Numerical,Potential predictor after type validation
8,amount_usd,Numerical,Potential predictor
9,fee,Numerical,Potential predictor


## 6. Initial Observations

The initial data understanding exercise produced the following observations:

1. The dataset contains 11,400 transaction records and 26 variables covering transaction, customer, payment, device, geographic, behavioural, and risk-related information.

2. The target variable, `is_fraud`, is a binary variable containing only 0 and 1 values, with no missing observations. It is therefore suitable as the target variable for the supervised classification problem.

3. `transaction_id` is not unique. There are 11,200 unique transaction IDs across 11,400 records, resulting in 200 duplicated transaction ID occurrences. These records require further investigation during the data cleaning and profiling stages to determine whether they represent genuine repeated transactions or duplicate records.

4. The dataset contains 1,315 unique customers, 2,113 unique devices, and 10,900 unique IP addresses. These identifiers provide opportunities for understanding customer and behavioural relationships and may support the development of aggregated or historical features.

5. Several variables require data-type validation. In particular, `timestamp` is currently stored as an object rather than a datetime variable, while `amount_src` is also stored as an object despite representing a transaction amount.

6. Missing values are present in several variables, including `timestamp`, `amount_usd`, `fee`, `ip_address`, `ip_country`, `kyc_tier`, and `device_trust_score`. These missing values will be investigated systematically during data profiling and data cleaning.

7. Variables such as `new_device`, `location_mismatch`, `ip_risk_score`, `device_trust_score`, transaction velocity measures, `corridor_risk`, and other risk-related variables appear potentially relevant to fraud detection and will be investigated further.

8. Identifier variables such as `transaction_id`, `customer_id`, `device_id`, and `ip_address` should not automatically be used as direct model predictors. Their potential value will instead be considered in relation to feature engineering and transaction-level analysis.

9. `risk_score_internal` requires particular attention during subsequent analysis because its suitability as a predictor depends on how and when the score is generated. If it incorporates information that would only become available after a fraud investigation or fraud outcome, its use could introduce data leakage.

## 7. Data Understanding Summary

The FinLora dataset provides a suitable starting point for developing a supervised machine learning fraud detection system. It contains transaction-level information across financial, customer, temporal, geographic, device, behavioural, and risk dimensions, together with a binary fraud outcome.

The initial assessment confirms that the dataset contains potentially valuable predictors for fraud classification. However, several data quality and structural issues require further investigation before modelling, including duplicated transaction identifiers, missing values, inappropriate data types, and the potential for data leakage in certain risk-related variables.

The findings from this notebook will inform the Data Profiling and Data Cleaning stages of the project. No permanent data modifications are performed in this notebook, as its purpose is to establish an initial understanding of the supplied data.